In [1]:
from app.models import Embedding, SubTopic, Bookmark, Document_Entity_Link, Entity, Document
from app.db_connector import get_engine
import app.getters as getter
import app.icog_util as util
from app.icog_util import DocSummarizer
from app.prompt_models import ResponseWithIndex, DocumentPromptVerbatim
from app.transformers_util import generate_embeddings, get_util
from app.together_api_client import TogetherMixtralClient
from sqlalchemy.orm import Session
from sqlalchemy import (
    text,
    and_, or_, select
)

engine = get_engine()
user_id = 'HqAXhad3jrUWmPibnMf1xZczNIq2'

client = TogetherMixtralClient()
summarizer = DocSummarizer()

2024-06-19 21:36:29 - Connecting to database using TCP
2024-06-19 21:36:29 - Connected to database using TCP


[nltk_data] Downloading package punkt to /home/eboraks/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


2024-06-19 21:36:33 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2024-06-19 21:36:33 - Starting new HTTPS connection (1): huggingface.co:443
2024-06-19 21:36:33 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-06-19 21:36:34 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json HTTP/1.1" 200 0
2024-06-19 21:36:34 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md HTTP/1.1" 200 0
2024-06-19 21:36:34 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-06-19 21:36:34 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json HTTP/1.1" 200 0
2024-06-19 21:36:34 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/co

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [2]:
docs = getter.get_documents_by_user_id(user_id)

In [3]:
d1 = docs[8]
sentences = util.original_text_to_sentences(summarizer(d1.original_text).toStr())
new_text = util.sentences_to_text(sentences)

In [4]:
res = await client.generate(DocumentPromptVerbatim, DocumentPromptVerbatim.get_messages(new_text))

2024-06-19 21:36:40 - Attempt 1 to generate summary
2024-06-19 21:36:49 - Response status: finished
2024-06-19 21:36:49 - Answer is not None. Stop retrying. Number of attempts 1


In [5]:
print(res.whatThisArticleIsAbout.answer)
print(res.whatThisArticleIsAbout.sentences_indcies) 
for ind in res.whatThisArticleIsAbout.sentences_indcies:
    print(sentences[ind])

This article is a comprehensive biography of Taylor Alison Swift, an American singer-songwriter who has significantly influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy.
[0, 1]
Taylor Alison Swift (born December 13, 1989) is an American singer-songwriter.
A subject of widespread public interest with a vast fanbase, she has influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy.


In [6]:
res


DocumentPromptVerbatim(whatThisArticleIsAbout=ResponseWithIndex(answer='This article is a comprehensive biography of Taylor Alison Swift, an American singer-songwriter who has significantly influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy.', sentences_indcies=[0, 1]), learningsFromTheArticle=ResponseWithIndex(answer="The article provides an in-depth look at Swift's career, including her early years, rise to fame, music styles, awards, and advocacy efforts.", sentences_indcies=[2, 3]), summaryInBulletPoints=[ResponseWithIndex(answer='Taylor Alison Swift is an American singer-songwriter who has significantly influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy.', sentences_indcies=[0, 1]), ResponseWithIndex(answer="Swift's career began in 2005 when she signed with Big Machine Records and achieved prominence as a country pop singer with th

In [7]:
print(res.learningsFromTheArticle.answer)
print(res.learningsFromTheArticle.sentences_indcies)
for ind in res.learningsFromTheArticle.sentences_indcies:
    print(sentences[ind])

The article provides an in-depth look at Swift's career, including her early years, rise to fame, music styles, awards, and advocacy efforts.
[2, 3]
She signed with Big Machine Records in 2005 and achieved prominence as a country pop singer with the albums Taylor Swift (2006) and Fearless (2008).
Their singles "Teardrops on My Guitar", "Love Story", and "You Belong with Me" were crossover successes on country and pop radio formats and brought Swift mainstream fame.


In [9]:
for point in res.summaryInBulletPoints:
    print(point.answer)
    print(point.sentences_indcies)
    for ind in point.sentences_indcies:
        print(sentences[ind])
    print('---')

Taylor Alison Swift is an American singer-songwriter who has significantly influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy.
[0, 1]
Taylor Alison Swift (born December 13, 1989) is an American singer-songwriter.
A subject of widespread public interest with a vast fanbase, she has influenced the music industry, popular culture, and politics through her songwriting, artistry, entrepreneurship, and advocacy.
---
Swift's career began in 2005 when she signed with Big Machine Records and achieved prominence as a country pop singer with the albums 'Taylor Swift' and 'Fearless'.
[2, 3]
She signed with Big Machine Records in 2005 and achieved prominence as a country pop singer with the albums Taylor Swift (2006) and Fearless (2008).
Their singles "Teardrops on My Guitar", "Love Story", and "You Belong with Me" were crossover successes on country and pop radio formats and brought Swift mainstream fame.
---
Swift has exp